# Data Mapping Inspection
Let's check if the image/edit_image mapping is correct in our dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# Load the datasets
train_df = pd.read_csv('data/example_image_dataset/metadata_edit.csv')
val_df = pd.read_csv('data/example_image_dataset/validation.csv')

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nFirst few training samples:")
print(train_df.head())

In [ ]:
def show_sample_comparison(df, idx, title_prefix=""):
    """Show image vs edit_image for a given sample"""
    row = df.iloc[idx]
    
    # Load images
    image_path = os.path.join('data/example_image_dataset', row['image'])
    edit_image_path = os.path.join('data/example_image_dataset', row['edit_image'])
    
    if not os.path.exists(image_path) or not os.path.exists(edit_image_path):
        print(f"Missing files for sample {idx}")
        return
    
    img1 = Image.open(image_path)
    img2 = Image.open(edit_image_path)
    
    # Create subplot
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(img1)
    axes[0].set_title(f'{title_prefix}image (column 1)\n{row["image"]}')
    axes[0].axis('off')
    
    axes[1].imshow(img2)
    axes[1].set_title(f'{title_prefix}edit_image (column 2)\n{row["edit_image"]}')
    axes[1].axis('off')
    
    plt.suptitle(f'Sample {idx}: "{row["prompt"]}"', fontsize=12, y=0.95)
    plt.tight_layout()
    plt.show()
    
    print(f"\nPrompt: {row['prompt']}")
    print(f"Analysis: If this prompt is asking to ADD something, then:")
    print(f"- The LEFT image should be the BEFORE (without the feature)")
    print(f"- The RIGHT image should be the AFTER (with the feature)")
    print("="*80)

In [ ]:
# Let's look at the specific faux fur collar sample from validation
# Find the sample with "Add a detachable faux fur collar for extra warmth and style variety"
fur_sample = val_df[val_df['prompt'].str.contains('Add a detachable faux fur collar for extra warmth and style variety')]
if len(fur_sample) > 0:
    idx = fur_sample.index[0]
    print(f"Found faux fur collar sample at validation index {idx}")
    show_sample_comparison(val_df, idx, "VALIDATION - ")
else:
    print("Faux fur collar sample not found in validation")

In [ ]:
# Let's look at some training samples with "Add" prompts
add_samples = train_df[train_df['prompt'].str.contains('Add', case=False)]
print(f"Found {len(add_samples)} training samples with 'Add' in prompt")

# Show first few "Add" samples
for i in range(min(3, len(add_samples))):
    orig_idx = add_samples.index[i]
    print(f"\n=== Training Sample {orig_idx} ===")
    show_sample_comparison(train_df, orig_idx, "TRAINING - ")

In [ ]:
# Let's also look at some "Remove" prompts to see the pattern
remove_samples = train_df[train_df['prompt'].str.contains('Remove', case=False)]
print(f"Found {len(remove_samples)} training samples with 'Remove' in prompt")

# Show first few "Remove" samples
for i in range(min(2, len(remove_samples))):
    orig_idx = remove_samples.index[i]
    print(f"\n=== Training Sample {orig_idx} (Remove) ===")
    show_sample_comparison(train_df, orig_idx, "TRAINING - ")

In [ ]:
# Summary analysis
print("SUMMARY ANALYSIS:")
print("="*50)
print("Based on the images above, determine:")
print("1. When prompt says 'Add X', which image actually HAS the X?")
print("   - Left (image column) or Right (edit_image column)?")
print("")
print("2. When prompt says 'Remove X', which image is MISSING the X?")
print("   - Left (image column) or Right (edit_image column)?")
print("")
print("EXPECTED MAPPING for Qwen-Image-Edit:")
print("- image (column 1) = TARGET (what we want to generate)")
print("- edit_image (column 2) = INPUT (what we start with)")
print("- extra_inputs: 'edit_image' means use column 2 as input")
print("")
print("So for 'Add faux fur collar':")
print("- edit_image should be WITHOUT collar (input)")
print("- image should be WITH collar (target)")